# Semantic Chunking via Sentence Embeddings

Static character or token chunking frequently splits sentences or paragraphs in half, creating context fragmentation. Semantic chunking leverages sentence-level embeddings (all-MiniLM-L6-v2) to measure the cosine distance between adjacent sentences and introduces chunk boundaries exclusively where topic shifts occur.

## Workflow Architecture

<div align="center">
  <img src="workflow_semantic_chunking.png" alt="Semantic Chunking via Sentence Embeddings Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["Input Document"]
        Text(["Input Document Text"]):::startNode
    end
    subgraph Sentences["Sentence Tokenization"]
        SentTokenizer["Sentence Tokenizer"]:::sentNode
        S1["Sentence 1"]:::sNode
        S2["Sentence 2"]:::sNode
        S3["Sentence 3"]:::sNode
    end
    subgraph Similarity["Semantic Embedding and Distance"]
        Embedder["Sentence Transformer<br/>all-MiniLM-L6-v2"]:::embNode
        CosDist["Cosine Distance Calculation"]:::distNode
        Threshold{"Distance Exceeds Threshold?"}:::decideNode
    end
    subgraph Chunks["Dynamic Chunk Creation"]
        ChunkA["Chunk A: Sentences 1 and 2<br/>(High Semantic Coherence)"]:::chunkNode
        Boundary["Semantic Breakpoint Marker"]:::breakNode
        ChunkB["Chunk B: Sentence 3<br/>(Topic Shift)"]:::chunkNode
    end
    Text --> SentTokenizer
    SentTokenizer --> S1
    SentTokenizer --> S2
    SentTokenizer --> S3
    S1 --> Embedder
    S2 --> Embedder
    S3 --> Embedder
    Embedder --> CosDist
    CosDist --> Threshold
    Threshold -->|No: Same Topic| ChunkA
    Threshold -->|Yes: Topic Shift| Boundary
    Boundary --> ChunkB
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef sentNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef sNode fill:#F1F8E9,stroke:#558B2F,stroke-width:2px,color:#33691E;
    classDef embNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef distNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef decideNode fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#880E4F;
    classDef chunkNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef breakNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Chunking Principles
- **Meaning-Driven Boundaries**: Splits documents according to semantic shifts rather than arbitrary character counts.
- **Adaptive Chunk Sizing**: Chunks expand or contract organically based on paragraph thematic density.
- **Embedding-Driven Similarity**: Uses sentence-transformers MiniLM to determine semantic cohesion.


In [36]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9694.87it/s]


In [37]:
# Split the text into sentences
sentences = [s.strip() for s in text.split('\n') if s.strip()]
sentences

['LangChain is a framework for building applications with LLMs.',
 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents, memory, and retrievers.',
 'The Eiffel Tower is located in Paris.',
 'France is a popular tourist destination.']

In [38]:
embeddings = model.encode(sentences)
embeddings

array([[-0.02109226, -0.04472176,  0.01087079, ..., -0.01217804,
         0.08605649,  0.02890729],
       [-0.0341802 , -0.10210426,  0.00366992, ..., -0.01398786,
         0.04454359,  0.00551358],
       [-0.02442169, -0.05424953, -0.13623357, ...,  0.0365635 ,
         0.07216297, -0.03104779],
       [ 0.06605352,  0.03884848,  0.01661562, ...,  0.03093833,
         0.07990999,  0.05157555],
       [ 0.10403015, -0.03097693,  0.02524883, ...,  0.07805594,
         0.01353772, -0.02684898]], shape=(5, 384), dtype=float32)

In [39]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

threshold = 0.7
chunks = []             # This will hold all of our final chunks
current_chunk = []      # This holds the sentences for the chunk currently being built

# Edge case: If the document is empty
if not sentences:
    pass
else:
    # Initialize the first chunk with the very first sentence
    current_chunk.append(sentences[0])

    # Loop through the sentences, comparing sentence i with sentence i+1
    for i in range(len(sentences) - 1):
        
        # sklearn's cosine_similarity expects 2D arrays, so we reshape 1D embeddings
        emb_current = np.array(embeddings[i]).reshape(1, -1)
        emb_next = np.array(embeddings[i+1]).reshape(1, -1)
        
        sim = cosine_similarity(emb_current, emb_next)[0][0]
        
        if sim >= threshold:
            # The next sentence is similar enough; add it to the ongoing chunk
            current_chunk.append(sentences[i+1])
        else:
            # The context shifted. Save the current chunk, and start a new one.
            chunks.append(" ".join(current_chunk)) # Join list of sentences into a single string
            current_chunk = [sentences[i+1]]       # Start new chunk with the next sentence

    # if current_chunk is not empty then Don't forget to append the very last chunk after the loop finishes!
    if current_chunk:
        chunks.append(" ".join(current_chunk))

# Print the final grouped chunks
for i, c in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(c)
    print("\n")

--- Chunk 1 ---
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


--- Chunk 2 ---
You can create chains, agents, memory, and retrievers.


--- Chunk 3 ---
The Eiffel Tower is located in Paris.


--- Chunk 4 ---
France is a popular tourist destination.




In [40]:
class ThresholdSematicChunker:

    def __init__(self, model_name = 'all-MiniLM-L6-v2', threshold = .7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str):
        sentences = [s.strip() for s in text.split('.') if s.split()]
        embeddings = self.model.encode(sentences)
        chunk = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i-1], embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunk.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunk.append(". ".join(current_chunk) + ".")
        return chunk

    def split_documents(self, docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result

# chunking 

chunker = ThresholdSematicChunker(threshold=0.7)
chunks = chunker.split(text)
chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8473.34it/s]


['LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.']

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_groq import ChatGroq

# 1. FIXED: Use LangChain's specific wrapper for HuggingFace models
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# 1. Convert your raw strings into LangChain Document objects
document_chunks = [Document(page_content=chunk) for chunk in chunks]

# 2. Now pass them into FAISS
vectorstore = FAISS.from_documents(document_chunks, embedding_model)
retriever = vectorstore.as_retriever()

template = """Answer the question based on the following context:
{context}
Question: {question}
"""
prompt = PromptTemplate.from_template(template)

# Note: Make sure your GROQ_API_KEY environment variable is set
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

# 3. FIXED: Cleaned up the chain using modern LCEL syntax
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. FIXED: Because we used RunnablePassthrough(), you can just pass a string directly
query = "What is LangChain used for?"
result = rag_chain.invoke(query)

print(result)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12774.66it/s]


LangChain is a framework used for building applications that work with large language models (LLMs). It offers modular abstractions to combine LLMs with tools such as OpenAI and Pinecone, enabling developers to create chains, agents, memory components, and retrievers.


In [44]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.documents import Document

# 1. Build a document from the sample text
text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""
docs = [Document(page_content=text)]

# 2. Initialize the Hugging Face embedding model 
# This will download the model to your local machine the first time you run it
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Initialize the Semantic Chunker using the local Hugging Face model
chunker = SemanticChunker(embedding_model)

# 4. Split the document and print the results
chunks = chunker.split_documents(docs)

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("\n")

C:\Users\itsar\AppData\Local\Temp\ipykernel_31920\3449491567.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6964.36it/s]


--- Chunk 1 ---

LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.


--- Chunk 2 ---





In [50]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# 1. Build a document from the sample text
text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""
docs = [Document(page_content=text)]

# Create semantic chunks from the source document.
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
semantic_chunks = semantic_chunker.split_documents(docs)
semantic_chunker = SemanticChunker(embedding_model)

# Build a vector store from the semantic chunks.
vectorstore = FAISS.from_documents(semantic_chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}
Answer:"""
)

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

rag_chain = ({"context": retriever | (lambda documents: "\n\n".join(document.page_content for document in documents)),"question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "What is LangChain used for?"
answer = rag_chain.invoke(query)
print(answer)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2035.19it/s]


LangChain is a framework for building applications with large language models (LLMs). It provides modular abstractions that let you combine LLMs with tools such as OpenAI and Pinecone, and enables you to create chains, agents, memory, and retrievers.
